## Convoluted Neural Network (CNN) - `keras`

All training code called here.

- Training Time Taken
- Test Accuracy

For Keras, the documentation is pretty rich, and gives good insight into how to use the api. This is a good starting reference. [https://keras.io/examples/vision/image_classification_from_scratch/](https://keras.io/examples/vision/image_classification_from_scratch/)

### Image Loading

There is difference in how you load images for linear classifiers as opposed to convoluted networks. I had spent time to understand the difference on loading images for `LinearSVC` and `MLPCLassifier` vs a Convolution Neural Network. For CNN training, it's assumed you have a large dataset, way more than what can be loaded in memory, to manage that, Keras supports options to load and classify data from a directory. Since have a small dataset, we can use it off memory.

#### `numpy`

A good amount of time was spent trying to make sure the data was available in a format which is "trainable" for the model. Reading a couple of chapters on `numpy.shape` and `numpy.reshape` helped with the transformation to prepare the data for training.


### Output Details

Results for neural network are captured in `/results/cnn_results_{yyyymmddhhmm()}.csv`


In [7]:
import datetime, base64, io, os, errno, math

import keras
import platform, sys

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from PIL import Image

from time import perf_counter
import tensorflow as tf

from keras.layers import Dense, Dropout, Activation, Flatten
from keras.layers import Conv2D, MaxPool2D, AveragePooling2D
from keras.models import Sequential

In [8]:
print(f"Architecture: {platform.machine()}")
print(f"Python Version: {sys.version.split()[0]}")
print(f"TensorFlow Version: {tf.__version__}")

gpus = tf.config.list_physical_devices("GPU")
print("GPUs found:", gpus)


def yyyymmddhhmm():
    return datetime.datetime.now().strftime("%Y%m%d%H%M")

Architecture: arm64
Python Version: 3.11.17
TensorFlow Version: 2.16.2
GPUs found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


### Loading STL-10 Data


In [9]:
# 5000 images training dataset
# path to the binary train file with image data
TRAIN_DATA_PATH = "./stl-10/train_X.bin"
# path to the binary train file with labels
TRAIN_LABEL_PATH = "./stl-10/train_y.bin"
# 8000 images training dataset
# path to the binary test file with image data
TEST_DATA_PATH = "./stl-10/test_X.bin"
# path to the binary test file with labels
TEST_LABEL_PATH = "./stl-10/test_y.bin"

# path to the binary test file with labels
UNLABELED = "./stl-10/unlabeled_X.bin"

# STL-10 dimensions: 96x96 pixels, 3 channels (RGB)
HEIGHT = 96
WIDTH = 96
CHANNELS = 3

NUM_CLASSES = 10

# size of a single image in bytes
SIZE = HEIGHT * WIDTH * CHANNELS
INPUT_SHAPE = (HEIGHT, WIDTH, CHANNELS)


def load_stl10_binary(data_path, label_path=None):

    # Read images (stored in column-major order)
    with open(data_path, "rb") as f:
        everything = np.fromfile(f, dtype=np.uint8)
        # Images are stored as: channels, width, height
        images = np.reshape(everything, (-1, CHANNELS, WIDTH, HEIGHT))
        # Transpose to Keras format: (batch, height, width, channels)
        images = np.transpose(images, (0, 3, 2, 1))

    if label_path:
        with open(label_path, "rb") as f:
            labels = np.fromfile(f, dtype=np.uint8)
        # Convert 1-indexed (1-10) to 0-indexed (0-9) for Keras cross-entropy
        labels = labels - 1
        return images, labels

    return images


# Usage
X_train, y_train = load_stl10_binary(TRAIN_DATA_PATH, TRAIN_LABEL_PATH)
X_test, y_test = load_stl10_binary(TEST_DATA_PATH, TEST_LABEL_PATH)
# X_unlabeled = load_stl10_binary(UNLABELED)

print("Train shape:", X_train.shape)  # Output: (5000, 96, 96, 3)

# Rescaling
X_train = X_train.astype("float32") / 255
X_test = X_test.astype("float32") / 255


# One hot encoding labels, everything is mutable...
y_train = tf.keras.utils.to_categorical(y_train, num_classes=10)
y_test = tf.keras.utils.to_categorical(y_test, num_classes=10)

print(y_train[:1])

Train shape: (5000, 96, 96, 3)
[[0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]]


### Training Levers

We can train the model by changing parameters listed below


In [10]:
RESNET50_POOLING_AVERAGE = "avg"
DENSE_LAYER_ACTIVATION = "softmax"
OBJECTIVE_FUNCTION = "categorical_crossentropy"

BATCH_SIZE = 128
LEARNING_RATE = 1e-4

FACTOR = 2
LAYERS = NUM_CLASSES * FACTOR

# Common accuracy metric for all outputs, but can use different metrics for different output
LOSS_METRICS = ["accuracy"]

N_JOBS = 5  # parallel processing of jobs
RESULTS_CSV = f"./results/cnn_results_{yyyymmddhhmm()}.csv"

### Training a Regular CNN

[Keras Link](https://keras.io/getting_started/intro_to_keras_for_engineers/)

Using Sequential API


In [11]:
def modelling(density, enable_callbacks, lr, epochs, batch_size):

    model_name = f"relu_{density}_{lr}_{epochs}_{batch_size}_{yyyymmddhhmm()}"
    output = f"results/cnn_{model_name}.csv"

    model = Sequential(
        [
            keras.Input(shape=INPUT_SHAPE),
            keras.layers.Conv2D(
                filters=32, kernel_size=(3, 3), padding="same", activation="relu"
            ),
            keras.layers.BatchNormalization(),
            keras.layers.Conv2D(
                filters=32, kernel_size=(3, 3), padding="same", activation="relu"
            ),
            keras.layers.BatchNormalization(),
            keras.layers.MaxPooling2D(pool_size=(30, 30)),
            keras.layers.Flatten(),
            keras.layers.Dropout(0.25),
            keras.layers.Dense(256, activation="relu"),
            keras.layers.Dropout(0.5),

            keras.layers.BatchNormalization(),
            keras.layers.Dense(128, activation="relu"),
            keras.layers.Dropout(0.5),

            keras.layers.BatchNormalization(),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dropout(0.5),

            keras.layers.BatchNormalization(),
            keras.layers.Dense(NUM_CLASSES, activation="softmax"),
        ]
    )

    model.compile(
        loss=keras.losses.CategoricalCrossentropy(),
        optimizer=keras.optimizers.Adam(learning_rate=lr, weight_decay=1e-6),
        metrics=[keras.metrics.CategoricalAccuracy()],
    )

    callbacks = [
        keras.callbacks.CSVLogger(
            filename=output,
            separator=",",
        ),
    ]

    model.summary()

    history = model.fit(
        X_train,
        y_train,
        batch_size=batch_size,
        epochs=epochs,
        shuffle=True,
        validation_split=0.15,
        callbacks=callbacks,
    )
    model.summary()

    model.save(f"{model_name}.keras")

    print(history)

    return model

#### Fit & Accuracy Test


In [12]:
# model = modelling(NUM_CLASSES * 1, True, 1e-2)
# score = model.evaluate(X_test, y_test)
# print(score)


# model = modelling(NUM_CLASSES * 2, True, 1e-2)
# score = model.evaluate(X_test, y_test)
# print(score)

# model = modelling(NUM_CLASSES, True, 1e-3, 50, 128)
# score = model.evaluate(X_test, y_test)
# print(score)


model = modelling(NUM_CLASSES, True, 1e-4, 300, 64)
score = model.evaluate(X_test, y_test)
print(score)

# modelling(NUM_CLASSES * 2, True, 1e-2)
# modelling(NUM_CLASSES * 4, True, 1e-4)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 96, 96, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 96, 96, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 96, 96, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 96, 96, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 3, 3, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 288)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 288)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 256)            │        73,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 127,978 (499.91 KB)

 Trainable params: 126,954 (495.91 KB)

 Non-trainable params: 1,024 (4.00 KB)

Epoch 1/300
67/67 ━━━━━━━━━━━━━━━━━━━━ 15s 154ms/step - categorical_accuracy: 0.1104 - loss: 2.9598 - val_categorical_accuracy: 0.1093 - val_loss: 2.3450
Epoch 2/300
67/67 ━━━━━━━━━━━━━━━━━━━━ 8s 125ms/step - categorical_accuracy: 0.1155 - loss: 2.8803 - val_categorical_accuracy: 0.1093 - val_loss: 2.3621
Epoch 3/300
67/67 ━━━━━━━━━━━━━━━━━━━━ 8s 123ms/step - categorical_accuracy: 0.1108 - loss: 2.8135 - val_categorical_accuracy: 0.1093 - val_loss: 2.3779
Epoch 4/300
67/67 ━━━━━━━━━━━━━━━━━━━━ 8s 121ms/step - categorical_accuracy: 0.1207 - loss: 2.7274 - val_categorical_accuracy: 0.1067 - val_loss: 2.3736
Epoch 5/300
67/67 ━━━━━━━━━━━━━━━━━━━━ 8s 122ms/step - categorical_accuracy: 0.1224 - loss: 2.7309 - val_categorical_accuracy: 0.1053 - val_loss: 2.3345
Epoch 6/300
67/67 ━━━━━━━━━━━━━━━━━━━━ 8s 126ms/step - categorical_accuracy: 0.1492 - loss: 2.6073 - val_categorical_accuracy: 0.1053 - val_loss: 2.2843
Epoch 7/300
67/67 ━━━━━━━━━━━━━━━━━━━━ 8s 125ms/step - categorical_accuracy: 0.14

... (3 outputs left out)

 Trainable params: 126,954 (495.91 KB)

 Non-trainable params: 1,024 (4.00 KB)

 Optimizer params: 253,910 (991.84 KB)

250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - categorical_accuracy: 0.6026 - loss: 1.0943
[1.0943175554275513, 0.6026250123977661]


### ResNet-50: Transfer Learning

A transfer learning model is a machine learning technique where a model developed for a source task is reused as the starting point for a different, related target task


In [13]:
def resNet50():
    input_tensor = keras.Input((224, 224, 3))
    up_sample = (224, 224)

    model_name = f"resNet50_{2e-5}_{yyyymmddhhmm()}"
    output = f"results/cnn_{model_name}.csv"

    res50 = keras.applications.ResNet50(include_top=False, weights="imagenet", input_tensor=(input_tensor))

    for layer in res50.layers[:143]:
        layer.trainable = False

    for i, layer in enumerate(res50.layers):
        print(i, layer.name, "-", layer.trainable)

    model = Sequential(
        [
            keras.layers.Lambda(lambda image: tf.image.resize(image, up_sample)),
            res50,
            keras.layers.Flatten(),
            keras.layers.BatchNormalization(),
            keras.layers.Dense(256, activation="relu"),
            keras.layers.Dropout(0.5),
            keras.layers.BatchNormalization(),
            keras.layers.Dense(128, activation="relu"),
            keras.layers.Dropout(0.5),
            keras.layers.BatchNormalization(),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dropout(0.5),
            keras.layers.BatchNormalization(),
            keras.layers.Dense(10, activation="softmax"),
        ]
    )

    model.compile(
        loss=keras.losses.CategoricalCrossentropy(),
        optimizer=keras.optimizers.Adam(learning_rate=2e-5),
        metrics=[keras.metrics.CategoricalAccuracy()],
    )

    # Instead of tuning things by hand, use some factors and math to decide when to stop, based on callbacks!
    callbacks = [
        keras.callbacks.CSVLogger(
            filename=output,
            separator=",",
        )
    ]

    model.summary()

    history = model.fit(
        X_train,
        y_train,
        batch_size=BATCH_SIZE,
        epochs=25,
        validation_data=(X_test, y_test),
        callbacks=[callbacks],
    )
    model.summary()

    print(history)

In [14]:
# resNet50()